# 03 JUNE 2026

In [1]:
import pandas as pd
import numpy as np
import re
import emoji
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
import torch
from torch import nn,optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report

path = r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\train.csv"

df_train = pd.read_csv(path)
df_train.drop(['id'], axis=1,inplace=True)

lemmatizer = WordNetLemmatizer()
sw = stopwords.words('english')
    
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|https\S+|www\S+','',text)
    text = re.sub(r"<.*?>",'',text)
    text = re.sub(r'@\w+|#\w+','',text)
    text = re.sub(r'(.)\1{2,}',r'\1\1',text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = emoji.replace_emoji(text, '')
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in sw]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

df_train['cleaned_comment'] = df_train['comment_text'].apply(clean_text)
X = df_train['cleaned_comment']
df_train['toxicity_ind'] = df_train[['toxic',
                         'severe_toxic',
                         'obscene',
                         'threat',
                         'insult',
                         'identity_hate']].sum(axis=1)
df_train['toxicity_ind'] = df_train['toxicity_ind'].apply(lambda x: 1 if x > 0 else 0)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = 30000,
    output_sequence_length =100,
    output_mode = 'int' 
)
vectorizer.adapt(X.values)
vectorized_text = vectorizer(X.values)
vocab_size = len(vectorizer.get_vocabulary())

class SimpleLSTM(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size,output_size):
        super(SimpleLSTM,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=embed_size)
        self.lstm = nn.LSTM(embed_size,hidden_size,batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size,output_size)
        
    def forward(self,X):
        X = self.embedding(X)
        out,(ht, ct) =self.lstm(X)
        out = ht[-1]
        out = self.dropout(out)
        out = self.fc(out)
        return out

X_train, X_val, y_train, y_val = train_test_split(
    vectorized_text.numpy(),
    df_train['toxicity_ind'].values,
    test_size=0.2,
    random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
y_val = torch.tensor(y_val, dtype=torch.float32).reshape(-1,1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(
    val_dataset,
    batch_size=128
)

model = SimpleLSTM(vocab_size, embed_size=128, hidden_size=128, output_size=1)

positive_count = sum(df_train['toxicity_ind'])
negative_count = len(df_train) - positive_count
pos_weight = torch.tensor([negative_count / positive_count])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)
for epoch in range(10):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        output = model(batch_X)
        loss = criterion(output, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

val_loss = 0
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for batch_X, batch_y in val_loader:
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        val_loss += loss.item()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

avg_val_loss = val_loss / len(val_loader)
all_preds = np.array(all_preds).flatten()
all_labels = np.array(all_labels).flatten()

print(
    f"Epoch {epoch+1} "
    f"Train Loss:{avg_loss:.4f} "
    f"Val Loss:{avg_val_loss:.4f}"
)

print(classification_report(
        all_labels,
        all_preds,
        digits=4
    )
)

Epoch 1, Average Loss: 1.1558
Epoch 2, Average Loss: 0.7689
Epoch 3, Average Loss: 0.5017
Epoch 4, Average Loss: 0.3776
Epoch 5, Average Loss: 0.3111
Epoch 6, Average Loss: 0.2627
Epoch 7, Average Loss: 0.2186
Epoch 8, Average Loss: 0.1864
Epoch 9, Average Loss: 0.1584
Epoch 10, Average Loss: 0.1383
Epoch 10 Train Loss:0.1383 Val Loss:0.6757
              precision    recall  f1-score   support

         0.0     0.9813    0.9511    0.9659     28671
         1.0     0.6600    0.8394    0.7389      3244

    accuracy                         0.9397     31915
   macro avg     0.8206    0.8952    0.8524     31915
weighted avg     0.9486    0.9397    0.9429     31915



In [2]:
import pandas as pd
import numpy as np
import re
import emoji
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
import torch
from torch import nn,optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

path = r"C:\Users\nisha\OneDrive\Desktop\Nisha\GUVI\MiniProject\CommentToxicity\Data\train.csv"

df_train = pd.read_csv(path)
df_train.drop(['id'], axis=1,inplace=True)

lemmatizer = WordNetLemmatizer()
sw = stopwords.words('english')
    
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|https\S+|www\S+','',text)
    text = re.sub(r"<.*?>",'',text)
    text = re.sub(r'@\w+|#\w+','',text)
    text = re.sub(r'(.)\1{2,}',r'\1\1',text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = emoji.replace_emoji(text, '')
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in sw]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

df_train['cleaned_comment'] = df_train['comment_text'].apply(clean_text)
X = df_train['cleaned_comment']
df_train['toxicity_ind'] = df_train[['toxic',
                         'severe_toxic',
                         'obscene',
                         'threat',
                         'insult',
                         'identity_hate']].sum(axis=1)
df_train['toxicity_ind'] = df_train['toxicity_ind'].apply(lambda x: 1 if x > 0 else 0)

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = 30000,
    output_sequence_length =100,
    output_mode = 'int' 
)
vectorizer.adapt(X.values)
vectorized_text = vectorizer(X.values)
vocab_size = len(vectorizer.get_vocabulary())

class SimpleLSTM(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size,output_size):
        super(SimpleLSTM,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=embed_size,padding_idx=0)
        self.lstm = nn.LSTM(embed_size,hidden_size,num_layers=1,batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_size,output_size)
        
    def forward(self,X):
        X = self.embedding(X)
        out,(ht, ct) =self.lstm(X)
        out = ht[-1]
        out = self.dropout(out)
        out = self.fc(out)
        return out

X_train, X_val, y_train, y_val = train_test_split(
    vectorized_text.numpy(),
    df_train['toxicity_ind'].values,
    test_size=0.2,
    random_state=42,
    stratify=df_train['toxicity_ind']
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
y_val = torch.tensor(y_val, dtype=torch.float32).reshape(-1,1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(
    val_dataset,
    batch_size=128
)

model = SimpleLSTM(vocab_size, embed_size=64, hidden_size=128, output_size=1)

positive_count = sum(df_train['toxicity_ind'])
negative_count = len(df_train) - positive_count
pos_weight = torch.tensor([negative_count / positive_count])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=1e-5)
best_val_loss = float('inf')
patience = 3
counter = 0

for epoch in range(10):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        output = model(batch_X)
        loss = criterion(output, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    #print(f"Epoch {epoch+1}, Average Loss: {avg_loss:.4f}")

    val_loss = 0
    all_preds = []
    all_labels = []
    model.eval()
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()

    print(
            f"Epoch {epoch+1} "
            f"Train Loss:{avg_loss:.4f} "
            f"Val Loss:{avg_val_loss:.4f}"
        )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(
            model.state_dict(),
            "best_bilstm_model.pth"
        )
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

    for threshold in [0.3, 0.4, 0.5, 0.6]:
        preds = (all_preds > threshold).astype(int)
        print(f"\nThreshold = {threshold}")
        print(
            classification_report(
                all_labels,
                preds,
                digits=4
            )
        )
        print(
            "Confusion Matrix:\n",
            confusion_matrix(all_labels, preds)
        )

Epoch 1 Train Loss:1.2234 Val Loss:1.1599

Threshold = 0.3
              precision    recall  f1-score   support

         0.0     1.0000    0.0002    0.0003     28670
         1.0     0.1017    1.0000    0.1846      3245

    accuracy                         0.1018     31915
   macro avg     0.5508    0.5001    0.0925     31915
weighted avg     0.9087    0.1018    0.0191     31915

Confusion Matrix:
 [[    5 28665]
 [    0  3245]]

Threshold = 0.4
              precision    recall  f1-score   support

         0.0     0.9738    0.0311    0.0602     28670
         1.0     0.1039    0.9926    0.1881      3245

    accuracy                         0.1288     31915
   macro avg     0.5388    0.5118    0.1242     31915
weighted avg     0.8853    0.1288    0.0732     31915

Confusion Matrix:
 [[  891 27779]
 [   24  3221]]

Threshold = 0.5
              precision    recall  f1-score   support

         0.0     0.9239    0.9148    0.9193     28670
         1.0     0.3073    0.3341    0.3201 